Сохранение submissions тут закомментировано.
Если нужно запускать именно с целью посмотреть на них - раскомментировать последние строки в соответствующих ячейках.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
np.random.seed(42)

In [3]:
dataset = pd.read_csv('Run200_Wave_0_1.txt', sep=' ', header=None, skipinitialspace=True)
dataset = dataset.drop([0, 1, 2, 3, 504], axis=1)
dataset.columns = list(range(500))

In [4]:
# Feature Engineering v2 (charge comparison)
def extract_features_v2(signal):
    s = 2**14 - np.array(signal) - 1560
    features = {}
    features['baseline'] = np.mean(s[:50])
    s_corrected = s - features['baseline']
    features['peak_amplitude'] = np.max(s_corrected)
    peak_idx = np.argmax(s_corrected)
    features['peak_position'] = peak_idx
    threshold_10 = 0.1 * features['peak_amplitude']
    threshold_90 = 0.9 * features['peak_amplitude']
    rise_start = np.where(s_corrected[:peak_idx] >= threshold_10)[0]
    rise_end = np.where(s_corrected[:peak_idx] >= threshold_90)[0]
    features['rise_time'] = (rise_end[0] - rise_start[0]) if len(rise_start) > 0 and len(rise_end) > 0 else 0
    fall_start = np.where(s_corrected[peak_idx:] <= threshold_90)[0]
    fall_end = np.where(s_corrected[peak_idx:] <= threshold_10)[0]
    features['fall_time'] = (fall_end[0] - fall_start[0]) if len(fall_start) > 0 and len(fall_end) > 0 else 0
    total_integral = np.sum(s_corrected)
    fast_window = 10
    fast_start = max(0, peak_idx - fast_window)
    fast_end = min(500, peak_idx + fast_window)
    fast_integral = np.sum(s_corrected[fast_start:fast_end])
    tail_start = peak_idx + fast_window
    tail_integral = np.sum(s_corrected[tail_start:])
    features['fast_ratio'] = fast_integral / (total_integral + 1)
    features['tail_ratio'] = tail_integral / (total_integral + 1)
    try:
        fall_region = s_corrected[peak_idx + fall_start[0] : peak_idx + fall_end[0] + 1]
        if len(fall_region) > 5:
            x = np.arange(len(fall_region))
            y = np.log(fall_region + 1)
            slope, _ = np.polyfit(x, y, 1)
            features['decay_tau'] = -1.0 / (slope + 1e-10)
        else:
            features['decay_tau'] = 0
    except:
        features['decay_tau'] = 0
    features['skewness'] = stats.skew(s_corrected)
    features['asymmetry'] = np.sum(s_corrected[:peak_idx]) / (np.sum(s_corrected[peak_idx:]) + 1)
    features['width_50'] = np.sum(s_corrected > 0.5 * features['peak_amplitude'])
    return features

features_list_v2 = []
for i in range(len(dataset)):
    features_list_v2.append(extract_features_v2(dataset.iloc[i]))
df_features_v2 = pd.DataFrame(features_list_v2)
df_features_v2 = df_features_v2.replace([np.inf, -np.inf], np.nan).fillna(0)

selected_v2 = ['peak_amplitude', 'peak_position', 'rise_time', 'tail_ratio', 
               'decay_tau', 'width_50', 'asymmetry', 'baseline']
X_v2 = df_features_v2[selected_v2].copy()
X_scaled_v2 = StandardScaler().fit_transform(X_v2)

In [5]:
# Feature Engineering v3 (дробные интегралы хвоста)
def extract_features_v3(signal):
    s = 2**14 - np.array(signal) - 1560
    features = {}
    baseline = np.mean(s[:50])
    s_corrected = s - baseline
    peak_idx = np.argmax(s_corrected)
    peak_amplitude = np.max(s_corrected)
    features['peak_amplitude'] = peak_amplitude
    features['peak_position'] = peak_idx
    total = np.sum(s_corrected)
    fast = np.sum(s_corrected[max(0,peak_idx-10):min(500,peak_idx+10)])
    tail = np.sum(s_corrected[peak_idx+10:])
    features['tail_ratio'] = tail / (total + 1)
    features['tail_10_30'] = np.sum(s_corrected[peak_idx+10:peak_idx+30]) / (total + 1)
    features['tail_30_60'] = np.sum(s_corrected[peak_idx+30:peak_idx+60]) / (total + 1)
    features['tail_60_100'] = np.sum(s_corrected[peak_idx+60:peak_idx+100]) / (total + 1)
    features['tail_10_60'] = np.sum(s_corrected[peak_idx+10:peak_idx+60]) / (total + 1)
    features['ratio_early_late'] = (features['tail_10_30'] + 1e-10) / (features['tail_60_100'] + 1e-10)
    try:
        fall_region = s_corrected[peak_idx+5:peak_idx+40]
        if len(fall_region) > 5:
            x = np.arange(len(fall_region))
            y = np.log(fall_region + 1e-10)
            slope, _ = np.polyfit(x, y, 1)
            features['decay_tau'] = -1.0 / (slope + 1e-10)
        else:
            features['decay_tau'] = 0
    except:
        features['decay_tau'] = 0
    features['baseline'] = baseline
    features['width_50'] = np.sum(s_corrected > 0.5 * peak_amplitude)
    features['asymmetry'] = np.sum(s_corrected[:peak_idx]) / (np.sum(s_corrected[peak_idx:]) + 1)
    return features

features_v3 = []
for i in range(len(dataset)):
    features_v3.append(extract_features_v3(dataset.iloc[i]))
df_v3 = pd.DataFrame(features_v3)
df_v3 = df_v3.replace([np.inf, -np.inf], 0).fillna(0)

# Подбор веса decay_tau (двумерное правило), cont=0.10

In [6]:
cont = 0.10
iso = IsolationForest(n_estimators=300, contamination=cont, random_state=42)
anomaly_mask = iso.fit_predict(X_scaled_v2) == -1

tail_norm = X_v2['tail_ratio'].values[~anomaly_mask]
decay_norm = X_v2['decay_tau'].values[~anomaly_mask]

print(f"{'Вес':<8s} {'Порог':<10s} {'Кл.0':<8s} {'Кл.1':<8s} {'Кл.2':<8s}")
print("-" * 45)

for weight in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    combined = tail_norm + weight * decay_norm
    thr = np.median(combined)
    
    labels = np.zeros(len(X_v2), dtype=int)
    combined_all = X_v2['tail_ratio'].values + weight * X_v2['decay_tau'].values
    labels[~anomaly_mask] = (combined_all[~anomaly_mask] > thr).astype(int)
    labels[anomaly_mask] = 2
    
    u, c = np.unique(labels, return_counts=True)
    c_dict = dict(zip(u, c))
    
    print(f"{weight:<8.2f} {thr:<10.4f} {c_dict.get(0,0):<8d} {c_dict.get(1,0):<8d} {c_dict.get(2,0):<8d}")
    
    mapping = {0: 1, 1: 0, 2: 2}
    labels_perm = np.array([mapping[l] for l in labels])
    sub = pd.DataFrame({'index': range(len(labels_perm)), 'cluster': labels_perm})
    #sub.to_csv(f'submission_2d_w{int(weight*100)}.csv', index=False)
    #print(f"  Сохранён: submission_2d_w{int(weight*100)}.csv")

Вес      Порог      Кл.0     Кл.1     Кл.2    
---------------------------------------------
0.05     0.4081     10566    10565    2348    
0.08     0.5222     10566    10565    2348    
0.10     0.5980     10566    10565    2348    
0.12     0.6731     10566    10565    2348    
0.15     0.7868     10566    10565    2348    
0.20     0.9747     10566    10565    2348    


Подбор веса decay_tau, но cont=0.09

In [7]:
cont = 0.09
iso = IsolationForest(n_estimators=300, contamination=cont, random_state=42)
anomaly_mask = iso.fit_predict(X_scaled_v2) == -1

tail_norm = X_v2['tail_ratio'].values[~anomaly_mask]
decay_norm = X_v2['decay_tau'].values[~anomaly_mask]

print(f"{'Вес':<8s} {'Порог':<10s} {'Кл.0':<8s} {'Кл.1':<8s} {'Кл.2':<8s}")
print("-" * 45)

for weight in [0.05, 0.10, 0.15, 0.20]:
    combined = tail_norm + weight * decay_norm
    thr = np.median(combined)
    
    labels = np.zeros(len(X_v2), dtype=int)
    combined_all = X_v2['tail_ratio'].values + weight * X_v2['decay_tau'].values
    labels[~anomaly_mask] = (combined_all[~anomaly_mask] > thr).astype(int)
    labels[anomaly_mask] = 2
    
    u, c = np.unique(labels, return_counts=True)
    c_dict = dict(zip(u, c))
    
    print(f"{weight:<8.2f} {thr:<10.4f} {c_dict.get(0,0):<8d} {c_dict.get(1,0):<8d} {c_dict.get(2,0):<8d}")
    
    mapping = {0: 1, 1: 0, 2: 2}
    labels_perm = np.array([mapping[l] for l in labels])
    sub = pd.DataFrame({'index': range(len(labels_perm)), 'cluster': labels_perm})
    #sub.to_csv(f'submission_2d_cont09_w{int(weight*100)}.csv', index=False)
    #print(f"  Сохранён: submission_2d_cont09_w{int(weight*100)}.csv")

Вес      Порог      Кл.0     Кл.1     Кл.2    
---------------------------------------------
0.05     0.4083     10683    10682    2114    
0.10     0.5982     10683    10682    2114    
0.15     0.7872     10683    10682    2114    
0.20     0.9754     10683    10682    2114    


Подбор веса decay_tau, но cont=0.08

In [8]:
cont = 0.08
iso = IsolationForest(n_estimators=300, contamination=cont, random_state=42)
anomaly_mask = iso.fit_predict(X_scaled_v2) == -1

tail_norm = X_v2['tail_ratio'].values[~anomaly_mask]
decay_norm = X_v2['decay_tau'].values[~anomaly_mask]

print(f"{'Вес':<8s} {'Порог':<10s} {'Кл.0':<8s} {'Кл.1':<8s} {'Кл.2':<8s}")
print("-" * 45)

for weight in [0.10, 0.15, 0.20, 0.25, 0.30]:
    combined = tail_norm + weight * decay_norm
    thr = np.median(combined)
    
    labels = np.zeros(len(X_v2), dtype=int)
    combined_all = X_v2['tail_ratio'].values + weight * X_v2['decay_tau'].values
    labels[~anomaly_mask] = (combined_all[~anomaly_mask] > thr).astype(int)
    labels[anomaly_mask] = 2
    
    u, c = np.unique(labels, return_counts=True)
    c_dict = dict(zip(u, c))
    
    print(f"{weight:<8.2f} {thr:<10.4f} {c_dict.get(0,0):<8d} {c_dict.get(1,0):<8d} {c_dict.get(2,0):<8d}")
    
    mapping = {0: 1, 1: 0, 2: 2}
    labels_perm = np.array([mapping[l] for l in labels])
    sub = pd.DataFrame({'index': range(len(labels_perm)), 'cluster': labels_perm})
    #sub.to_csv(f'submission_cont08_w{int(weight*100)}.csv', index=False)
    #print(f"  Сохранён: submission_cont08_w{int(weight*100)}.csv")

Вес      Порог      Кл.0     Кл.1     Кл.2    
---------------------------------------------
0.10     0.5986     10800    10800    1879    
0.15     0.7876     10800    10800    1879    
0.20     0.9756     10800    10800    1879    
0.25     1.1632     10800    10800    1879    
0.30     1.3513     10800    10800    1879    


Лучшие результаты на Kaggle дали такие пары:
- cont=0.10, вес=0.20
- cont=0.09, вес=0.20
- cont=0.08, вес=0.20

Оптимальный вес decay_tau ≈ 0.20

# Поиск лучшей пары признаков (v3)

In [9]:
# Поиск лучшей пары признаков из v3
from itertools import combinations

selected_v3 = ['tail_ratio', 'decay_tau', 'tail_10_30', 'tail_30_60', 
               'tail_10_60', 'ratio_early_late', 'asymmetry', 'peak_amplitude', 'width_50']

X_v3 = df_v3[selected_v3].copy()
X_scaled_v3 = StandardScaler().fit_transform(X_v3)

cont = 0.08
iso = IsolationForest(n_estimators=300, contamination=cont, random_state=42)
anomaly_mask = iso.fit_predict(X_scaled_v3) == -1

print(f"Аномалий: {anomaly_mask.sum()} ({anomaly_mask.sum()/len(anomaly_mask)*100:.1f}%)\n")
print(f"{'Признаки':<35s} {'Separation':<12s}")
print("-" * 50)

best_sep_overall = 0
best_combo = None

for feat1, feat2 in combinations(['tail_ratio', 'decay_tau', 'tail_10_30', 'tail_10_60', 'ratio_early_late'], 2):
    v1_norm = X_v3[feat1].values[~anomaly_mask]
    v2_norm = X_v3[feat2].values[~anomaly_mask]
    
    for w in [0.1, 0.5, 1.0, 2.0]:
        combined = v1_norm + w * v2_norm
        thr = np.median(combined)
        
        g0 = combined[combined <= thr]
        g1 = combined[combined > thr]
        if len(g0) > 100 and len(g1) > 100:
            sep = abs(g0.mean() - g1.mean()) / (g0.std() + g1.std() + 1e-10)
            
            if sep > best_sep_overall:
                best_sep_overall = sep
                best_combo = (feat1, feat2, w)

    if best_combo:
        print(f"{feat1} + {w:.1f}*{feat2} — sep={sep:.4f}")

print(f"\nЛучшая комбинация: {best_combo[0]} + {best_combo[2]:.1f}*{best_combo[1]} (separation={best_sep_overall:.4f})")

# Сохраняем лучшую комбинацию
feat1, feat2, w = best_combo
combined_all = X_v3[feat1].values + w * X_v3[feat2].values
combined_norm = combined_all[~anomaly_mask]
thr = np.median(combined_norm)

labels = np.zeros(len(X_v3), dtype=int)
labels[~anomaly_mask] = (combined_all[~anomaly_mask] > thr).astype(int)
labels[anomaly_mask] = 2

mapping = {0: 1, 1: 0, 2: 2}
labels_perm = np.array([mapping[l] for l in labels])

print(f"Распределение: 0={sum(labels_perm==0)}, 1={sum(labels_perm==1)}, 2={sum(labels_perm==2)}")

#sub = pd.DataFrame({'index': range(len(labels_perm)), 'cluster': labels_perm})
#sub.to_csv('submission_v3_best_pair.csv', index=False)

Аномалий: 1879 (8.0%)

Признаки                            Separation  
--------------------------------------------------
tail_ratio + 2.0*decay_tau — sep=1.7749
tail_ratio + 2.0*tail_10_30 — sep=2.4191
tail_ratio + 2.0*tail_10_60 — sep=2.5831
tail_ratio + 2.0*ratio_early_late — sep=0.3295
decay_tau + 2.0*tail_10_30 — sep=1.7775
decay_tau + 2.0*tail_10_60 — sep=1.7805
decay_tau + 2.0*ratio_early_late — sep=0.4375
tail_10_30 + 2.0*tail_10_60 — sep=2.6409
tail_10_30 + 2.0*ratio_early_late — sep=0.3317
tail_10_60 + 2.0*ratio_early_late — sep=0.3311

Лучшая комбинация: tail_10_30 + 2.0*tail_10_60 (separation=2.6409)
Распределение: 0=10800, 1=10800, 2=1879


Лучшая комбинация: tail_10_30 + 2.0*tail_10_60 (separation=2.6499).

Сабмишен submission_v3_best_pair.csv (порог = медиана):
Score на Kaggle = 0.80493.

Почему не остановились здесь?
Медиана делит данные пополам, но реальная физическая граница между гамма и нейтронами
может быть смещена. В эталонном ноутбуке дополнительно заменена медиана
на оптимальный порог (максимизирующий separation) — это дало микро-прирост
до 0.80569 - предпоследний submission.

Сам Optimal threshold — в финальном ноутбуке (final_solution.ipynb).